# 9. 요청의 강도와 Mandate: 모델이 승인해도 실행되지 않는 경우

원본: https://github.com/deepseek-ai/dsh-superagent-prototype/tree/8c32a9dc5d72c033816c6a8110b37f40cf94fba4

근거 파일: `src/policy.ts`: settingsPolicy, assessmentOutcome, policyVersion; `src/core/evals.ts`

**범위:** 원본은 TypeScript/Cordis입니다. 아래는 원본의 일부 불변조건을 Python/LangGraph로 실험하며 공식 API 호환 구현이 아닙니다. 새 커널에서 순서대로 실행합니다. 외부 API 호출이 없습니다.

## 목표: 판단 결과와 권한을 분리
원본의 gate는 explicit/inferred/ignore를 구분합니다. allowed sender가 직접 요청하면 explicit 경로이며 모든 규칙의 의미 평가가 필수인 경로는 아닙니다. inferred 후보는 활성 규칙 전체 평가, 명확한 구현 방향, intent evidence를 요구합니다. observe 모드는 would_accept만 기록합니다. 실행 중 정책 버전이 바뀌면 오래된 판단은 무효입니다.

아래 graph는 **inferred 후보 경로만** 모델링합니다. `assess`는 고정된 판단 대역이고 `validate`는 host의 결정적 검증입니다. LangGraph를 쓰는 위치는 우리 교육용 선택입니다.

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
class State(TypedDict):
    mode: str
    snapshot_version: int
    current_version: int
    matched: bool
    implementation_requested: bool
    target_settled: bool
    design_decision: bool
    evidence: str
    judgment: str
    outcome: str

def assess(s): return {'judgment':'accept'}
def validate(s):
    if s['snapshot_version']!=s['current_version']: return {'outcome':'unavailable:stale-policy'}
    if s['mode']=='off': return {'outcome':'ignore:disabled'}
    if not(s['implementation_requested'] and s['target_settled']) or s['design_decision']:
        return {'outcome':'ignore:direction-unsettled'}
    if not s['matched']: return {'outcome':'ignore:no-rule'}
    if not s['evidence']: raise ValueError('accept에는 intent evidence가 필요합니다')
    return {'outcome':'would_accept' if s['mode']=='observe' else 'accept'}
g=StateGraph(State)
g.add_node('assessment',assess); g.add_node('host_validation',validate)
g.add_edge(START,'assessment'); g.add_edge('assessment','host_validation'); g.add_edge('host_validation',END)
app=g.compile()
base={'mode':'observe','snapshot_version':1,'current_version':1,'matched':True,
      'implementation_requested':True,'target_settled':True,'design_decision':False,
      'evidence':'명시된 오탈자를 기존 표기에 맞춰 수정','judgment':'','outcome':''}
assert app.invoke(base)['outcome']=='would_accept'
assert app.invoke({**base,'mode':'auto'})['outcome']=='accept'
assert app.invoke({**base,'current_version':2})['outcome']=='unavailable:stale-policy'
assert app.invoke({**base,'design_decision':True})['outcome']=='ignore:direction-unsettled'
assert app.invoke({**base,'mode':'off'})['outcome']=='ignore:disabled'
print('모델 accept와 host 승인은 다릅니다:',app.invoke(base)['outcome'])

모델 accept와 host 승인은 다릅니다: would_accept


## 권한의 범위는 입력과 함께 커지지 않는다
Mandate는 승인 당시 scope, strength, actor, policy version, rules를 보존합니다. 새로운 메시지는 context가 될 수 있지만 scope를 자동 확장하지 않습니다. 원본의 scope 준수에는 모델 프롬프트도 참여하므로 아래 단순 집합 검사처럼 모든 의미적 scope가 코드로 판정되는 것은 아닙니다.

In [2]:
from dataclasses import dataclass
@dataclass(frozen=True)
class Mandate:
    scope: frozenset[str]
    policy_version: int
    strength: str
m=Mandate(frozenset({'fix-typo'}),1,'inferred')
new_message='같이 결제 API도 바꿔 주세요'
assert 'redesign-payment' not in m.scope
try:
    app.invoke({**base,'evidence':''})
except ValueError:
    print('근거 없는 accept 거부')
else: raise AssertionError('누락된 evidence가 통과함')
assert m.scope==frozenset({'fix-typo'})

근거 없는 accept 거부


## 과제와 해설
과제: 규칙 id 목록이 `copy, visual`인데 모델이 `copy`만 평가하면 거부하도록 확장하세요. 해설: 원본 assessmentOutcome은 활성 rule 각각이 정확히 한 번 등장하는지 검사합니다. 단순히 하나라도 matched라고 승인하면 누락된 규칙과 알 수 없는 규칙을 검증하지 못합니다. policyVersion은 모델 설정·concurrency 변경 모두를 포함하는 전체 설정 revision과도 다릅니다.